# NumCompute Stream Demo

This notebook demonstrates the streaming decision tree framework.

It covers:

1. Loading CSV data using `numcompute.io`
2. Splitting data into chunks
3. Incremental training using `.partial_fit()`
4. Comparing a single decision tree with a bagging ensemble
5. Logging and visualising streaming metrics


In [ ]:
import numpy as np

from numcompute.io import read_csv
from numcompute.preprocessing import StandardScaler
from numcompute.pipeline import Pipeline
from numcompute.tree import DecisionTreeClassifier
from numcompute.ensemble import EnsembleClassifier
from numcompute.stream import StreamTrainer
from numcompute.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
)


## 1. Load CSV data using custom I/O

The data is loaded from `demo/stream_data.csv` using `read_csv()` from `io.py`.


In [ ]:
DATA_PATH = "demo/stream_data.csv"

data = read_csv(
    DATA_PATH,
    dtype=float,
    skip_header=True,
)

X = data[:, :-1]
y = data[:, -1].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5 labels:", y[:5])


## 2. Split data into chunks

Each chunk simulates incoming streaming data.


In [ ]:
chunk_size = 5
chunks = []

for start in range(0, X.shape[0], chunk_size):
    end = start + chunk_size
    chunks.append((X[start:end], y[start:end]))

print("Number of chunks:", len(chunks))
print("First chunk X shape:", chunks[0][0].shape)
print("First chunk y shape:", chunks[0][1].shape)


## 3. Build streaming models

Both models use the same pipeline interface:

`StandardScaler.partial_fit()` → model `.partial_fit()`


In [ ]:
tree_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", DecisionTreeClassifier(
        max_depth=3,
        min_samples_split=2,
        max_features=None,
        criterion="gini",
    )),
])

ensemble_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", EnsembleClassifier(
        n_estimators=5,
        method="bagging",
        max_depth=3,
        min_samples_split=2,
        max_features=None,
        criterion="gini",
        random_state=42,
    )),
])

tree_trainer = StreamTrainer(tree_pipeline)
ensemble_trainer = StreamTrainer(ensemble_pipeline)


## 4. Train incrementally using `.partial_fit()`

Each chunk is processed one at a time.


In [ ]:
tree_logs = []
ensemble_logs = []

for X_chunk, y_chunk in chunks:
    tree_logs.append(tree_trainer.fit_score_chunk(X_chunk, y_chunk))
    ensemble_logs.append(ensemble_trainer.fit_score_chunk(X_chunk, y_chunk))

print("Final decision tree log:")
print(tree_logs[-1])

print("\nFinal ensemble log:")
print(ensemble_logs[-1])


## 5. Extract streaming metrics

In [ ]:
tree_accuracy = [log["cumulative_accuracy"] for log in tree_logs]
ensemble_accuracy = [log["cumulative_accuracy"] for log in ensemble_logs]

tree_error = [log["chunk_error"] for log in tree_logs]
ensemble_error = [log["chunk_error"] for log in ensemble_logs]

print("Tree cumulative accuracy:", tree_accuracy)
print("Ensemble cumulative accuracy:", ensemble_accuracy)


## 6. Visualise decision tree accuracy over time

In [ ]:
plot_metric_over_time(
    tree_accuracy,
    title="Decision Tree Cumulative Accuracy Over Time",
    ylabel="Accuracy",
)


## 7. Visualise decision tree error over time

In [ ]:
plot_metric_over_time(
    tree_error,
    title="Decision Tree Chunk Error Over Time",
    ylabel="Error",
)


## 8. Compare decision tree and bagging ensemble

In [ ]:
compare_models(
    tree_accuracy,
    ensemble_accuracy,
    labels=["Decision Tree", "Bagging Ensemble"],
    title="Streaming Model Comparison",
    ylabel="Cumulative Accuracy",
)


## 9. Predictions vs ground truth on latest chunk

In [ ]:
X_last, y_last = chunks[-1]
y_pred_last = ensemble_pipeline.predict(X_last)

plot_predictions_vs_ground_truth(
    y_last,
    y_pred_last,
    title="Latest Chunk: Predictions vs Ground Truth",
)


## Summary

This demo shows that the framework can:

- Load CSV data using the custom I/O module
- Process data chunk by chunk
- Train models incrementally using `.partial_fit()`
- Compare a single decision tree with a bagging ensemble
- Log and visualise streaming metrics
